# Scenario Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Summarize scenario impacts, driver sensitivities, fan charts, and variance decomposition for stress-review workflows.

**Prerequisites:** `06_scenarios/scenarios_and_stress_testing.ipynb`.

**What you'll learn:**

- Prepare scenario comparison data.
- Render `reporting.scenario_tearsheet` for stress-test review.
- Separate scenario calculation from presentation.

Stress a forecast four ways — a driver tornado, a base/upside/downside
comparison, a Monte-Carlo percentile fan, and a variance table — and render them
with `scenario_tearsheet`. Each input is produced by a real engine call.


In [ ]:
import datetime as dt
import json

import sys
sys.path.insert(0, "..")

from _shared import demo_pl_builder, demo_pl_model
from finstack_quant import reporting
from finstack_quant.statements import Evaluator, ForecastSpec, MonteCarloConfig
from finstack_quant.statements_analytics import (
    ScenarioSet,
    SensitivityConfig,
    VarianceConfig,
    evaluate_scenario_set,
    generate_tornado_entries,
    run_sensitivity,
    run_variance,
)

QUARTERS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]

def build(model_id, revenue, cogs, opex, *, stochastic_revenue=False):
    """A four-quarter cut of the shared P&L fixture, actuals through 2025Q2."""
    if stochastic_revenue:
        # Actuals only; forecast periods are sampled stochastically for the MC
        # fan, so the builder variant is used to attach the spec before build().
        builder = demo_pl_builder(
            model_id, periods=QUARTERS, revenue=revenue[:2], cogs=cogs, opex=opex, margins=False
        )
        builder.forecast("revenue", ForecastSpec.normal(mean=0.05, std_dev=0.15, seed=42))
        return builder.build()
    return demo_pl_model(
        model_id, periods=QUARTERS, revenue=revenue, cogs=cogs, opex=opex, margins=False
    )

base = build("scenario-base", [100, 110, 115, 120], [60, 65, 68, 72], [20, 22, 23, 24])
result = Evaluator().evaluate(base)
print("Base EBITDA 2025Q4:", result.get("ebitda", "2025Q4"))


## Driver tornado

`run_sensitivity` perturbs a forecast driver; `generate_tornado_entries` ranks
the impact on the target metric.

In [ ]:
sens_cfg = SensitivityConfig(
    "diagonal",
    [("revenue", "2025Q3", 115.0, [95.0, 105.0, 115.0, 125.0, 135.0])],
    ["ebitda"],
)
tornado_entries = generate_tornado_entries(run_sensitivity(base, sens_cfg), "ebitda", "2025Q3")
tornado = [json.loads(entry.to_json()) for entry in tornado_entries]
print("Tornado:", tornado)

## Scenario comparison

`evaluate_scenario_set` applies named overrides to the forecast periods; we read
the target metric from each scenario.

In [ ]:
scen_cfg = ScenarioSet({
    "upside": {"revenue": 135.0},
    "downside": {"revenue": 100.0},
})
scen_out = evaluate_scenario_set(base, scen_cfg)
scenarios = {"base": result.get("ebitda", "2025Q4")}
scenarios.update({name: scen_out.get(name).get("ebitda", "2025Q4") for name in scen_out.names})
print("Scenarios (EBITDA 2025Q4):", scenarios)

## Monte-Carlo fan

A stochastic revenue forecast (`ForecastSpec.normal`) drives the simulation; we
read the 5th / 50th / 95th percentiles per forecast period into a fan. The band
widens with the driver's volatility.

In [ ]:
mc_model = build("scenario-mc", [100, 110, 115, 120], [60, 65, 68, 72], [20, 22, 23, 24], stochastic_revenue=True)
mc = Evaluator().evaluate_monte_carlo(mc_model, MonteCarloConfig(2000, 42, [0.05, 0.5, 0.95]))
periods = mc.forecast_periods
p_low = mc.percentile_by_period("ebitda", 0.05)
p_mid = mc.percentile_by_period("ebitda", 0.5)
p_high = mc.percentile_by_period("ebitda", 0.95)
monte_carlo = {
    "periods": periods,
    "p_low": [p_low[p] for p in periods],
    "p_mid": [p_mid[p] for p in periods],
    "p_high": [p_high[p] for p in periods],
}
print("MC fan:", monte_carlo)

## Variance vs an alternate plan

In [ ]:
plan = build("scenario-plan", [105, 116, 124, 132], [62, 67, 70, 74], [21, 23, 24, 25])
plan_eval = Evaluator().evaluate(plan)
variance_report = run_variance(
    result,
    plan_eval,
    VarianceConfig("base", "plan", ["revenue", "ebitda"], ["2025Q3", "2025Q4"]),
)
variance = json.loads(variance_report.to_json())

## Forward-risk tear sheet

In [ ]:
presentation_report = reporting.scenario_tearsheet(
    tornado=tornado,
    scenarios=scenarios,
    monte_carlo=monte_carlo,
    variance=variance,
    target_metric="ebitda",
    title="Acme Corp — Forward Risk",
    generated=dt.date(2026, 6, 22),
)
# Replace the helper's static numeric-mode caption with provenance guidance.
presentation_report.meta_lines=["Source calculations determine arithmetic; display formatting only"]
presentation_report


## Saving a standalone HTML file

```python
ts = reporting.scenario_tearsheet(tornado=tornado, scenarios=scenarios, monte_carlo=monte_carlo, variance=variance, generated=dt.date(2026, 6, 22))
ts.save("scenario_tearsheet.html")
```

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
